# 背包问题

**类别：** 装箱

来源：[https://www.hexaly.com/templates/knapsack-problem](https://www.hexaly.com/templates/knapsack-problem)


## 问题

**背包问题**定义如下。考虑一组具有不同重量和价值的物品。我们需要选择一个物品子集放入一个已知容量的背包中。背包中物品的总重量不得超过其容量。目标是最大化背包中物品的总价值。

	

### 学到的建模原则

- 了解 Hexaly Optimizer 的建模风格：[区分决策变量与中间表达式](https://www.hexaly.com/docs/last/modelingprinciples/modelingprinciples.html#distinguish-decision-variables-from-intermediate-variables)


## 数据

我们提供来自 [OR Library](http://people.brunel.ac.uk/~mastjjb/jeb/info.html) 的实例。数据集的格式如下：

- 物品的数量
- 对每个物品，其重量
- 对每个物品，其价值
- 该实例的已知上界


## 模型

我们将背包问题建模为整数规划。对每个物品，我们定义一个 布尔决策变量，当物品被选中时为 1，否则为 0。我们使用 **sum** 算子计算被选中物品的总重量。注意，我们是从决策变量的值推导出这个值：总重量是一个中间表达式，而非决策变量。定义它之后，我们可以约束总重量小于背包的容量。类似地，我们定义另一个中间表达式，对应于被选中物品的总价值。最后，我们最大化该总价值。

尽管背包问题是 NP-hard 问题，但涉及数百万个物品的实例可以使用 Hexaly Optimizer 求解。


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys

if len(sys.argv) < 2:
    print("Usage: python knapsack.py inputFile [outputFile] [timeLimit]")
    sys.exit(1)


def read_integers(filename):
    with open(filename) as f:
        return [int(elem) for elem in f.read().split()]


with hexaly.optimizer.HexalyOptimizer() as optimizer:
    #
    # Read instance data
    #
    file_it = iter(read_integers(sys.argv[1]))

    # Number of items
    nb_items = next(file_it)

    # Items properties
    weights = [next(file_it) for i in range(nb_items)]
    values = [next(file_it) for i in range(nb_items)]

    # Knapsack bound
    knapsack_bound = next(file_it)

    #
    # Declare the optimization model
    #
    model = optimizer.model

    # Decision variables x[i]
    x = [model.bool() for i in range(nb_items)]

    # Weight constraint
    knapsack_weight = model.sum(x[i] * weights[i] for i in range(nb_items))
    model.constraint(knapsack_weight <= knapsack_bound)

    # Maximize value
    knapsack_value = model.sum(x[i] * values[i] for i in range(nb_items))
    model.maximize(knapsack_value)

    model.close()

    # Parameterize the optimizer
    if len(sys.argv) >= 4:
        optimizer.param.time_limit = int(sys.argv[3])
    else:
        optimizer.param.time_limit = 20

    optimizer.solve()

    #
    # Write the solution in a file
    #
    if len(sys.argv) >= 3:
        with open(sys.argv[2], 'w') as f:
            f.write("%d\n" % knapsack_value.value)
            for i in range(nb_items):
                if x[i].value != 1:
                    continue
                f.write("%d " % i)
            f.write("\n")
